# R0-v3: 0.8B SFT strengthening / SFT 补强

Date: 2026-11-13. Reproduction notebook; the agent-controlled run is separate.
Choose an L4 GPU. This reproduces the frozen data/code protocol from R0-v2, using a fresh optimizer and a separate output folder. It performs SFT and matched G8/MetricX diagnostics, with no DAPO updates.

在原 D 盘项目保存的 `r2_source_bundle.zip` 含冻结代码、清洗后的数据和 R0-v2 起点适配器。下方上传此包。Drive 挂载只用于此复现实验持久化，所有授权在 Google 页面处理。


In [ ]:
import pathlib, hashlib, zipfile, os, sys, subprocess, json, time
import torch
assert torch.cuda.is_available(), "Choose L4 GPU in Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))
from google.colab import files
uploaded=files.upload()
matches=[b for b in uploaded.values() if hashlib.sha256(b).hexdigest()=="3427808b038e896deb68f0c8841cd01672be765cbd1c0672d06ae4edcfe2d0f1"]
assert len(matches)==1, "Frozen source bundle SHA256 mismatch"
PROJECT=pathlib.Path('/content/r2_sft_reproduction')
PROJECT.mkdir(exist_ok=True)
bundle=PROJECT/'source_bundle.zip';bundle.write_bytes(matches[0])
with zipfile.ZipFile(bundle) as z:
    assert all(not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts for n in z.namelist())
    z.extractall(PROJECT)
os.chdir(PROJECT)
os.environ.update(HF_HOME=str(PROJECT/'.cache/huggingface'),TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1',OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='4')
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-colab.txt'],check=True)


In [ ]:
import importlib.metadata
from packaging.version import Version
try:
    if Version(importlib.metadata.version('torchao'))<Version('0.16.0'):
        subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
except importlib.metadata.PackageNotFoundError:
    pass
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3.5-0.8B',revision='2fc06364715b967f1860aea9cf38778875588b17',local_dir=PROJECT/'models/qwen3.5-0.8b',allow_patterns=['*.json','*.jinja','*.txt','*.safetensors','LICENSE'])
subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)
from google.colab import drive
drive.mount('/content/drive')
RUN=pathlib.Path('/content/drive/MyDrive/deep learning project')/('r2_reproduction_'+time.strftime('%Y%m%d_%H%M%S'))
RUN.mkdir(parents=True,exist_ok=False)
print("Separate run:",RUN)


In [ ]:
# For an interrupted reproduction, set RUN to its exact existing folder and add --resume.
subprocess.run([sys.executable,'-m','mtgrpo.r2_train','--run-dir',str(RUN),'--initial-adapter',str(PROJECT/'initial_adapter'),'--data-dir',str(PROJECT/'data'),'--microbatch','8'],check=True,cwd=PROJECT)
print(json.loads((RUN/'summary.json').read_text()))


In [ ]:
# Matched same-hardware/precision diagnostics. Numerical improvement requires semantic review.
snapshot_download('google/metricx-24-hybrid-large-v2p6',revision='51e875ba5c525c81627cfd135ee10f43c87dce00',local_dir=PROJECT/'models/metricx-24-hybrid-large',allow_patterns=['config.json','pytorch_model.bin','model.safetensors'])
snapshot_download('google/mt5-xl',revision='63fc6450d80515b48e026b69ef2fbbd426433e84',local_dir=PROJECT/'models/metricx-tokenizer',allow_patterns=['spiece.model','tokenizer*.json','special_tokens_map.json','config.json'])
subprocess.run([sys.executable,'r2_readiness.py','--run-root',str(RUN)],check=True,cwd=PROJECT)
print(json.loads((RUN/'readiness/summary.json').read_text()))


## Optional auxiliary controls / 可选辅助对照

Run only after the matched primary diagnostic finishes. Upload the four `r2_*_plan.json` files and four corresponding `r2_*_remote.py` files from `evidence/tools` in the complete package. Local CLI/authentication wrappers are unnecessary in this notebook. These cells add no SFT or DAPO updates.

主诊断完成后，从完整包的 `evidence/tools` 上传采样校准、末期检查点核查、奖励校准各自的 plan.json 和 remote.py，以及NLL完整扩展各自的plan/script，共8个小文件。无需本机CLI登录工具。仅把归档脚本的原执行路径显式适配到本次 `PROJECT`，保留原文件与改写后的哈希；`run` 链接指向这次独立的Drive目录，不覆盖旧模型。

The T0.7 control is adaptive, the16 stress cases are purposively selected development examples, and the semantic controls are AI-assisted. They do not become independent tests or prove DAPO improvement. Auxiliary exports are copied to the new Drive run and SHA-verified.


In [ ]:
import shutil
aux_uploaded = files.upload()
aux_names = [
    ("r2_sampling_calibration_plan.json", "sampling_calibration_plan.json", "r2_sampling_calibration_remote.py"),
    ("r2_last_checkpoint_probe_plan.json", "last_checkpoint_probe_plan.json", "r2_last_checkpoint_probe_remote.py"),
    ("r2_reward_calibration_plan.json", "reward_calibration_plan.json", "r2_reward_calibration_remote.py"),
    ("r2_nll_full_plan.json", "nll_full_plan.json", "r2_nll_full_remote.py"),
]
required = {x for original, renamed, script in aux_names for x in [original, script]}
assert required.issubset(aux_uploaded), "Upload the eight plan/script files listed above"
assert json.loads((RUN/'readiness/summary.json').read_text())["completed"]
link = PROJECT/'run'
if os.path.lexists(link):
    assert link.resolve() == RUN.resolve(), "Existing project/run points to a different run"
else:
    link.symlink_to(RUN, target_is_directory=True)
# This marker certifies only the already-finished primary diagnostic precondition.
# It is explicitly not a fabricated original CLI export manifest.
done = RUN/'runner_done.json'
if not done.exists():
    done.write_text(json.dumps({"reproduction_aux_precondition":True, "primary_readiness_completed":True}))
originals = PROJECT/'aux_originals'; originals.mkdir(exist_ok=True)
adaptations = {}
for original, renamed, script in aux_names:
    for name in [original, script]:
        target = originals/name
        if target.exists():
            assert target.read_bytes() == aux_uploaded[name], "Preserve prior auxiliary source"
        else:
            target.write_bytes(aux_uploaded[name])
    plan = PROJECT/renamed
    if plan.exists():
        assert plan.read_bytes() == aux_uploaded[original], "Do not mix different control plans"
    else:
        plan.write_bytes(aux_uploaded[original])
    text = aux_uploaded[script].decode('utf-8')
    original_path = "pathlib.Path('/content/r2_sft_project')"
    assert text.count(original_path) == 1
    adapted = text.replace(original_path, f"pathlib.Path({str(PROJECT)!r})")
    destination = PROJECT/('portable_'+script)
    destination.write_text(adapted, encoding='utf-8')
    adaptations[script] = {"original_sha256":hashlib.sha256(aux_uploaded[script]).hexdigest(),
                           "path_adapted_sha256":hashlib.sha256(destination.read_bytes()).hexdigest()}
(RUN/'auxiliary_path_adaptations.json').write_text(json.dumps(adaptations,indent=2))
print("Prepared path-only adaptations for", PROJECT)


In [ ]:
aux_exports = RUN/'auxiliary_exports'; aux_exports.mkdir(exist_ok=True)
for script, metadata in [
    ("r2_sampling_calibration_remote.py", "sampling_calibration/live.json"),
    ("r2_last_checkpoint_probe_remote.py", "last_checkpoint_probe/done.json"),
    ("r2_reward_calibration_remote.py", "reward_calibration/done.json"),
    ("r2_nll_full_remote.py", "nll_full/live.json"),
]:
    if script == "r2_nll_full_remote.py":
        full_plan_path=PROJECT/'nll_full_plan.json'
        full_plan=json.loads(full_plan_path.read_text())
        checkpoint=json.loads((PROJECT/'last_checkpoint_probe/identity.json').read_text())["checkpoint"]
        adaptation={"original_checkpoint":full_plan["checkpoint"],"reproduction_min_nll_checkpoint":checkpoint,
                    "reason":"Apply the unchanged minimum-NLL rule to this reproduction's own trained checkpoints"}
        full_plan["checkpoint"]=checkpoint
        full_plan_path.write_text(json.dumps(full_plan,indent=2))
        (RUN/'auxiliary_checkpoint_path_adaptation.json').write_text(json.dumps(adaptation,indent=2))
    subprocess.run([sys.executable,str(PROJECT/('portable_'+script))],cwd=PROJECT,check=True)
    info=json.loads((PROJECT/metadata).read_text())
    if "completed" in info:
        assert info["completed"]
    archive=pathlib.Path(info["path"])
    assert hashlib.sha256(archive.read_bytes()).hexdigest()==info["sha256"]
    saved=aux_exports/archive.name; shutil.copy2(archive,saved)
    assert hashlib.sha256(saved.read_bytes()).hexdigest()==info["sha256"]
    print("Verified Drive export:",saved)
print("No policy updates occurred in these auxiliary controls. Review source meaning before deciding readiness.")
